# wag 🐾 — generate on colab

the 4060 is the binding constraint at home: 8 GB total, and a 12B with a 2.6k prefix
spills to CPU and runs at **1.28 tok/s**. the only local model that fits comfortably is a
4B, which failed the structured tags on 4 of 7 rewrites. so generation moves here.

**this notebook adds no generation code.** vLLM serves an OpenAI-compatible endpoint on
localhost, and `gemini_convo.py fill --backend local` already speaks that — the same shim
that talked to LM Studio. prompts, parsing, marker quotas, resume logic and the shard
contract are all the repo's, unchanged. two drifting copies of the prompts is how you end
up with half a dataset in a different voice.

the flow is files, not networking:

    shards -> Drive -> colab generates -> Drive -> git pull at home

no tunnels, no ngrok, nothing to authenticate.

## pick a runtime first

| gpu | vram | what fits | notes |
|---|---:|---|---|
| **A100** | 40 GB | a 24B at bf16, or a 12B with room to spare | best quality per hour |
| **L4** | 24 GB | a 12B at bf16 comfortably | the sensible default |
| T4 | 16 GB | a 12B needs awq/gptq | no bf16 — turing. expect fiddling |

Runtime → Change runtime type → **L4** unless you want the 24B.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# vllm pulls its own torch. this takes a few minutes and is the slow part of the setup
!pip install -q vllm

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# ---- knobs -------------------------------------------------------------------
# the generator needs three things, in this order: it must reliably emit the
# <turn role="..."> tags, it must write decent prose, and it must not decline the
# intimate slice. a reasoning model is a bad fit — it emits thinking traces that
# parse_convo rejects outright.
MODEL   = "inflatebot/MN-12B-Mag-Mell-R1"   # rp-native merge, 12B, fits an L4 at bf16
MAXLEN  = 8192       # per-request window. the convo prefix alone is ~2.6k
PORT    = 8000

# set here rather than in the smoke-test cell: `!python` inherits os.environ, and a
# run that skips a cell shouldn't silently fall back to lm studio's default port
os.environ["LOCAL_BASE_URL"] = f"http://localhost:{PORT}/v1"

DRIVE   = Path("/content/drive/MyDrive/wag")
SHARDS  = DRIVE / "shards"          # put in_NNN.json here from home
REPO    = Path("/content/wag")
DRIVE.mkdir(parents=True, exist_ok=True)
SHARDS.mkdir(parents=True, exist_ok=True)

# the code comes from a tarball in Drive rather than a clone: the repo at home is ahead
# of its public remote and this work isn't published yet. drop a fresh wag-code.tar.gz in
# MyDrive/wag whenever the generators change. falls back to the clone if it isn't there.
REPO.mkdir(parents=True, exist_ok=True)
bundle = DRIVE / "wag-code.tar.gz"
if bundle.exists():
    !tar -xzf "{bundle}" -C {REPO}
    print("unpacked", bundle.name)
elif not (REPO / "gen_bulk.py").exists():
    !git clone -q https://github.com/Metrix187/wag {REPO}
    print("cloned from github (may be behind your local repo)")

need = ["gen_bulk.py", "gemini_convo.py", "gemini_rewrite.py", "slices.py",
        "data/anchors.md", "data/persona_spec.md"]
missing = [f for f in need if not (REPO / f).exists()]
if missing:
    raise SystemExit(f"bundle is missing {missing} — rebuild wag-code.tar.gz at home")
print("code ready:", len(list(REPO.glob("*.py"))), "modules")
print("shards waiting in Drive:", len(list(SHARDS.glob("in_*.json"))))

## the shards

copy `data/shards/in_*.json` from the repo at home into `MyDrive/wag/shards/`. if Drive
for Desktop is mounted that's just a file copy into `G:/My Drive/wag/shards/`. build them
first, at home:

```bash
python gemini_convo.py seed                # the ~5,100 net-new rows
python gen_bulk.py shard -n 6000 --size 45 # the rewrite half
python build_longinput.py                  # optional, §6
```

the cell below copies them into the repo checkout so the tools see them where they expect.

In [ ]:
dest = REPO / "data" / "shards"
dest.mkdir(parents=True, exist_ok=True)
n = 0
for p in sorted(SHARDS.glob("in_*.json")):
    shutil.copy2(p, dest / p.name)
    n += 1
# resume: anything already finished comes back too, so a re-run skips it
for p in sorted(SHARDS.glob("out_*.jsonl")):
    shutil.copy2(p, dest / p.name)

print(f"{n} input shards in place")
if not n:
    raise SystemExit("no in_*.json in MyDrive/wag/shards — copy them up first")

## serve it

vLLM in the background, then wait for the endpoint. the log goes to `/content/vllm.log` —
read it if the wait times out, it's where the OOM would be.

In [ ]:
# vllm brings its own torch (cuda 13.0) but colab ships torchaudio built for 12.8, and
# transformers imports torchaudio on the way in — the mismatch raises RuntimeError and
# kills the server before it binds. removing torchaudio makes transformers' availability
# check return False and skip the import entirely. nothing here wants audio.
!pip uninstall -q -y torchaudio

import urllib.request

log = open("/content/vllm.log", "w")
srv = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", str(PORT),
     "--max-model-len", str(MAXLEN),
     "--gpu-memory-utilization", "0.92",
     "--disable-log-requests"],
    stdout=log, stderr=subprocess.STDOUT,
)

t0 = time.time()
while time.time() - t0 < 1200:
    if srv.poll() is not None:
        print(open("/content/vllm.log").read()[-3000:])
        raise SystemExit("vllm died during startup — log above")
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5) as r:
            print(f"up in {time.time()-t0:.0f}s:",
                  [m["id"] for m in json.load(r)["data"]])
            break
    except Exception:
        time.sleep(10)
else:
    raise SystemExit("vllm never came up — check /content/vllm.log")

In [ ]:
# one real call through the repo's own shim, before committing to a long run.
# if the tags don't come back clean here they won't come back clean 5,000 times.
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import gemini_convo as gc, gemini_rewrite as gr

row = json.load(open(sorted((REPO/"data"/"shards").glob("in_*.json"))[0]))[0]
if row.get("kind") == "seed":
    got = gr._one_call(gr._client("local"), MODEL, gc.build_prefix(),
                       gc.build_prompt(row), 1.1)
    turns, why = gc.parse_convo(got["text"])
    print(f"parsed {len(turns)} turns" if turns else f"REJECTED: {why}")
    for m in turns[:4]:
        print(("  USER: " if m["role"] == "user" else "  WAG : ") + m["content"][:150])
else:
    got = gr._one_call(gr._client("local"), MODEL, gr.build_prefix(),
                       gr.build_prompt(row), 1.0)
    reply, think = gr.parse_reply(got["text"])
    print("reply:", reply[:300] if reply else "REJECTED: no <wag> block")

## generate

`--backend local` points the repo's shim at the vLLM server. resumable — re-run the cell
after a disconnect and it picks up from whatever's already in `out_*.jsonl`.

concurrency can go much higher than it could at home: vLLM batches properly, and the
window is per-request here rather than a slot carved out of a shared context.

In [ ]:
!python gemini_convo.py fill --shards all --backend local --model "{MODEL}" --concurrency 32

In [ ]:
# the rewrite half, if you sharded any. --candidates 3 writes cand_NNN.jsonl for a judge
# pass instead; judging locally needs a model that can follow the rubric, so that's
# usually better left for gemini
!python gemini_rewrite.py --all --backend local --model "{MODEL}" --concurrency 32

## back to Drive

do this **before** the runtime dies. colab will disconnect eventually and `/content` goes
with it.

In [ ]:
out = 0
for p in sorted((REPO / "data" / "shards").glob("out_*.jsonl")):
    shutil.copy2(p, SHARDS / p.name)
    out += 1
print(f"{out} out_*.jsonl -> {SHARDS}")

rows = ok = 0
for p in sorted(SHARDS.glob("out_*.jsonl")):
    for line in p.open(encoding="utf-8"):
        rows += 1
        ok += "error" not in json.loads(line)
print(f"{ok}/{rows} usable rows generated")

## then, at home

```bash
# copy out_*.jsonl from MyDrive/wag/shards into data/shards/
python gen_bulk.py merge
python gen_bulk.py filter
python gen_bulk.py sample -n 20     # READ THESE. twenty rows, by eye, every wave.
python gen_bulk.py build
```

`sample` is not optional. both bugs the first real run caught — a `heavy` row that was
mild grumbling about rain, and a grief conversation assigned `hmf` for mock indignation —
were invisible in the counts and obvious the moment anyone read a row.